In [0]:
%sql
-- Artists to promote based on popular genres in each country
-- "For each country, what is the most popular genre, and which artists are driving sales in that genre?"

WITH genre_revenue AS (

    SELECT
        c.Country,
        g.GenreName,
        SUM(il.UnitPrice * il.Quantity) AS genre_revenue
    FROM silver_invoiceline il
    JOIN silver_customer c
        ON il.CustomerId = c.CustomerId
    JOIN silver_track t
        ON il.TrackId = t.TrackId
    JOIN silver_genre g
        ON t.GenreId = g.GenreId
    GROUP BY c.Country, g.GenreName

),

top_genres AS (

    SELECT
        Country,
        GenreName,
        genre_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY Country
            ORDER BY genre_revenue DESC
        ) AS genre_rank
    FROM genre_revenue

),

artist_revenue AS (

    SELECT
        c.Country,
        g.GenreName,
        ar.ArtistName AS ArtistName,
        SUM(il.UnitPrice * il.Quantity) AS artist_revenue
    FROM silver_invoiceline il
    JOIN silver_customer c
        ON il.CustomerId = c.CustomerId
    JOIN silver_track t
        ON il.TrackId = t.TrackId
    JOIN silver_genre g
        ON t.GenreId = g.GenreId
    JOIN silver_album al
        ON t.AlbumId = al.AlbumId
    JOIN silver_artist ar
        ON al.ArtistId = ar.ArtistId
    GROUP BY
        c.Country,
        g.GenreName,
        ar.ArtistName

),

ranked_artists AS (

    SELECT
        Country,
        GenreName,
        ArtistName,
        artist_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY Country, GenreName
            ORDER BY artist_revenue DESC
        ) AS artist_rank
    FROM artist_revenue

)

SELECT
    tg.Country,
    tg.GenreName,
    ra.ArtistName,
    ROUND(ra.artist_revenue, 2) AS artist_revenue
FROM top_genres tg
JOIN ranked_artists ra
    ON tg.Country = ra.Country
   AND tg.GenreName = ra.GenreName
WHERE tg.genre_rank = 1
  AND ra.artist_rank <= 3
ORDER BY
    tg.Country,
    ra.artist_revenue DESC;